# Phase 5 · 05 — Residual bottleneck adapters, from scratch

Assignment 4 Q4.2:
`h' = h + W_up GELU(W_down h + b_down) + b_up`, bottleneck 32, one adapter on the
output of each block.

`W_up` and `b_up` are zero-initialised, so the adapted model **is** the frozen model
at step 0 — the cell asserts that on real logits before training starts. The
`enabled` flag is on each adapter, so a trained model can be switched off layer by
layer for the ablation study without retraining.

The thing to say about adapters in the report: unlike LoRA they cannot be merged
away, so the parameter saving does not buy an inference saving. `fig4` in the
analysis notebook shows that directly.

## 1 — Setup

In [ ]:
# ---- Google Colab: mount Drive ----
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, subprocess

# ---- EDIT: the Drive folder holding a1_tokenizer.py, bpe_40k.json, best_enc.pt ----
DRIVE = "/content/drive/MyDrive"

# ---- EDIT: your HuggingFace token (the dataset repo is gated) ----
HF_TOKEN = ""

# ---- which Phase 1 model to adapt ----
BACKBONE       = "encoder"      # "encoder" = best_enc.pt (MHA), "decoder" = best.pt (GQA)
DEC_CKPT_NAME  = "best.pt"      # the decoder checkpoint file name
MAX_LEN        = 512            # both Phase 1 models were pretrained at 512
ROPE_TABLE_LEN = 1024           # the decoder precomputes RoPE at 2 x max_seq_len

# ---- scale. QUICK=True is a ~20 min smoke test of the whole notebook. ----
QUICK          = False
# -----------------------------------------------------------------------------

WORK          = DRIVE                       # outputs -> {DRIVE}/cs781_phase5
OUT_DIR       = f"{WORK}/cs781_phase5"
TOK_JSON_PATH = f"{DRIVE}/bpe_40k.json"
TOK_PY_PATH   = f"{DRIVE}/a1_tokenizer.py"
ENC_PATH      = f"{DRIVE}/best_enc.pt"
DEC_PATH      = f"{DRIVE}/{DEC_CKPT_NAME}"

need = {"bpe_40k.json": TOK_JSON_PATH, "a1_tokenizer.py": TOK_PY_PATH,
        ("best_enc.pt" if BACKBONE == "encoder" else DEC_CKPT_NAME):
            (ENC_PATH if BACKBONE == "encoder" else DEC_PATH)}
for f, pth in need.items():
    ok = os.path.exists(pth)
    print(("FOUND    " if ok else "MISSING  ") + f
          + (f"  {os.path.getsize(pth)/1e6:.1f} MB" if ok else ""))
missing = [f for f, pth in need.items() if not os.path.exists(pth)]
if missing:
    raise FileNotFoundError(f"missing {missing} in {DRIVE} — fix DRIVE above")

if DRIVE not in sys.path:
    sys.path.insert(0, DRIVE)               # so `import a1_tokenizer` finds yours

for sub in ("cache", "models", "analysis", "plots", "submissions"):
    os.makedirs(f"{OUT_DIR}/{sub}", exist_ok=True)

try:                                        # preinstalled on Colab
    import huggingface_hub, sklearn, matplotlib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                           "huggingface_hub", "scikit-learn", "matplotlib"])

import torch
print(f"\noutputs -> {OUT_DIR}")
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("WARNING: no GPU — Runtime > Change runtime type > T4 GPU")

Mounted at /content/drive
FOUND    bpe_40k.json  1.2 MB
FOUND    a1_tokenizer.py  0.0 MB
FOUND    best_enc.pt  1565.3 MB

outputs -> /content/drive/MyDrive/cs781_phase5
torch 2.11.0+cu130
GPU: NVIDIA L4 (24 GB)


## 2 — Library

The next eight cells are **identical in every Phase 5 notebook**, so whichever one
you open is self-contained — there are no `.py` files to upload and no setup
notebook to run first.

Run them once per session (Runtime → Run all does it for you). You can collapse
them; nothing below depends on reading them.

The only file this still needs from your Drive is **`a1_tokenizer.py`**, your own
Phase 1 tokenizer module — it has to be yours so it matches `bpe_40k.json` exactly.

### Library 1/8 — imports, configuration, tokenizer

In [ ]:
import json, math, os, random, re, sys, time, zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LSIConfig:
    # --- paths (overridden in the notebook) ---
    drive_dir: str = "/content/drive/MyDrive"
    enc_ckpt: str = "/content/drive/MyDrive/best_enc.pt"
    dec_ckpt: str = "/content/drive/MyDrive/best.pt"
    tokenizer_json: str = "/content/drive/MyDrive/bpe_40k.json"
    out_dir: str = "/content/drive/MyDrive/cs781_phase5"
    cache_dir: str = "/content/drive/MyDrive/cs781_phase5/cache"
    results_json: str = "/content/drive/MyDrive/cs781_phase5/results.json"

    # --- document representation (held CONSTANT across the 8 experiments, so a
    #     difference in score is attributable to the adaptation technique and not
    #     to the window; the window itself is ablated separately in section 5) ---
    max_len: int = 512
    window: str = "head_tail"        # head | tail | head_tail
    head_frac: float = 0.5           # for head_tail: share of max_len from the start
    collapse_placeholders: bool = True   # see PLACEHOLDER_MAP — worth ~20% of the
                                         # token budget on this corpus

    # --- backbone ---
    n_heads: int = 12
    n_query_heads: int = 12          # decoder: query heads (kv inferred)
    rope_table_len: int | None = None
    dec_pool: str = "last"           # decoder readout: last | mean
    backbone: str = "encoder"        # encoder | decoder
    dropout: float = 0.1
    rope_theta: float = 10000.0

    # --- multi-label head / loss ---
    n_labels: int = 0                # filled in from the data
    loss: str = "bce"                # bce | bce_posw | focal | asl
    pos_weight_cap: float = 20.0     # cap on N_neg/N_pos so ultra-rare labels
                                     # cannot dominate the gradient
    focal_gamma: float = 2.0
    asl_gamma_neg: float = 4.0
    asl_gamma_pos: float = 1.0
    asl_clip: float = 0.05

    # --- training ---
    lr: float = 2e-5                 # backbone lr (the PEFT cells raise this)
    head_lr: float = 1e-3            # the classifier head always trains faster
    weight_decay: float = 0.01
    epochs: int = 3
    batch_size: int = 8
    grad_accum: int = 2
    warmup_frac: float = 0.06
    max_grad_norm: float = 1.0
    seed: int = 42
    eval_every: int = 300            # optimizer steps between dev evaluations
    patience: int = 5                # dev evals without macro-F1 improvement

    # --- decision rule ---
    min_labels: int = 1              # never emit an empty statute set
    thr_grid_lo: float = 0.02
    thr_grid_hi: float = 0.80
    thr_grid_n: int = 40
    per_label_min_pos: int = 8       # dev positives needed before a label gets its
                                     # own threshold (otherwise it keeps the global one)

    # --- data size (QUICK=True for a fast smoke run) ---
    quick: bool = False
    max_train: int | None = None
    max_dev: int | None = None

    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_token: str | None = None

    def __post_init__(self):
        if self.quick:
            self.max_train = self.max_train or 1500
            self.max_dev = self.max_dev or 500
            self.epochs = 1
            self.eval_every = 60
            self.patience = 3


def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD_NAMES = ["<pad>", "[PAD]", "<PAD>", "[pad]"]


def load_tokenizer(cfg: LSIConfig, py_dir: str | None = None):
    """Returns (tokenizer, pad_id, base_vocab_size). Identical to Phase 2-4."""
    d = py_dir or str(Path(cfg.tokenizer_json).parent)
    if d not in sys.path:
        sys.path.insert(0, d)
    from a1_tokenizer import BPETokenizer
    tok = BPETokenizer.load(cfg.tokenizer_json)
    base = int(tok.vocab_size)

    pad = None
    st = getattr(tok, "special_tokens", None)
    if isinstance(st, dict):
        for n in PAD_NAMES:
            if isinstance(st.get(n), int):
                pad = int(st[n]); break
    if pad is None:
        for attr in ("token_to_id", "vocab", "encoder", "stoi"):
            v = getattr(tok, attr, None)
            if isinstance(v, dict):
                for n in PAD_NAMES:
                    if isinstance(v.get(n), int):
                        pad = int(v[n]); break
            if pad is not None:
                break
    if pad is None:
        raise ValueError("Could not resolve the pad id from this tokenizer.")
    return tok, pad, base

### Library 2/8 — data loading, multi-hot labels, windowing, caching

The only change from Phase 4 is the label: `labels` is now a **list** of statute ids,
so each document becomes a multi-hot row of width `n_labels`.

In [ ]:
def _hf_path(cfg: LSIConfig, rel: str) -> str:
    from huggingface_hub import hf_hub_download
    return hf_hub_download(cfg.hf_repo, rel, repo_type="dataset", token=cfg.hf_token)


# ---- placeholder collapsing ----------------------------------------------
# The LSI corpus is anonymised: statute references, act names and party names
# are replaced by the literal strings <SECTION>, <ACT> and <ENTITY>. They are
# 5.5% of the words but the Phase 1 BPE has none of them as a single token, so
# they cost 4-5 tokens each against an average of 1.36 — measured on this
# training set, they eat 20.2% of every token window.
#
# Mapping each to a one-token symbol that IS in the Phase 1 vocabulary keeps the
# three markers distinct, needs no new embeddings (so the model is still exactly
# the Phase 1 model), and buys back about a quarter more real text inside the
# same 512 positions. The task statement lists preprocessing as a permitted
# method; section 5 measures whether it actually helps.
PLACEHOLDER_MAP = {"<SECTION>": "§", "<ACT>": "¶", "<ENTITY>": "@"}
_PH_RE = re.compile("|".join(re.escape(k) for k in PLACEHOLDER_MAP))


def normalize_text(text: str, collapse: bool = True) -> str:
    if not collapse:
        return text
    return _PH_RE.sub(lambda m: PLACEHOLDER_MAP[m.group(0)], text)


def load_split(cfg: LSIConfig, split: str, local: str | None = None):
    """split: train | dev | test. Returns list of (id, text, labels|None)."""
    if local and os.path.exists(local):
        path = local
    else:
        sub = "Test" if split == "test" else "Train"
        path = _hf_path(cfg, f"Phase_5_LSI/{sub}/{split}.json")
    with open(path) as f:
        raw = json.load(f)
    rows = raw.values() if isinstance(raw, dict) else raw
    out = []
    for r in rows:
        if not isinstance(r, dict):
            continue
        txt = r.get("text") or r.get("document") or ""
        if isinstance(txt, list):
            txt = " ".join(map(str, txt))
        if not txt:
            continue
        txt = normalize_text(txt, cfg.collapse_placeholders)
        lab = r.get("labels", r.get("label"))
        if lab is None:
            labs = None
        elif isinstance(lab, (list, tuple, set)):
            labs = sorted({int(x) for x in lab})
        else:
            labs = [int(lab)]
        out.append((str(r.get("id", len(out))), txt, labs))
    return out


def label_space(train_rows, dev_rows):
    """The statute id space. Ids are used as column indices directly, so the width
    is max_id + 1 — this is also how the evaluator derives the label dimension from
    its reference data, and it keeps a prediction a plain statute id with no
    remapping that could silently shift everything by one."""
    ids = set()
    for rows in (train_rows, dev_rows):
        for _, _, labs in rows:
            if labs:
                ids.update(labs)
    ids = sorted(ids)
    n = max(ids) + 1
    gaps = [i for i in range(n) if i not in set(ids)]
    print(f"statute ids: {len(ids)} distinct, min {min(ids)}, max {max(ids)} "
          f"-> label dimension {n}")
    if gaps:
        print(f"  {len(gaps)} id(s) in [0,{n-1}] never appear in train+dev: "
              f"{gaps[:12]}{' ...' if len(gaps) > 12 else ''}")
        print("  they stay as all-zero columns; a column the model can never get "
              "right still counts in macro-F1, so do not read a low macro-F1 as a "
              "bug without checking this first.")
    return ids, n


def multi_hot(rows, n_labels):
    Y = np.zeros((len(rows), n_labels), dtype=np.float32)
    for i, (_, _, labs) in enumerate(rows):
        if labs:
            Y[i, labs] = 1.0
    return Y


# ---- windowing: how a long judgment becomes max_len tokens ----

def encode_window(tok, text, pad_id, cfg: LSIConfig):
    """Returns (ids[max_len], n_real). Pre-truncates by words so the pure-Python
    BPE never processes material that would be discarded: encode() splits on
    whitespace and encodes each word independently, so the ids for the first N
    words are a prefix of the full document's ids (and likewise for a suffix)."""
    L, w = cfg.max_len, text.split()
    # Every word costs AT LEAST one BPE token, so the first L words already cover
    # an L-token window; 2L is a safety margin, and on this corpus (1.36 tokens
    # per word measured) it is a 2.7x one. The old 6L margin tripled the
    # tokenization work for nothing.
    budget_words = 2 * L

    if cfg.window == "head":
        ids = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)[:L]
    elif cfg.window == "tail":
        ids = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)[-L:]
    elif cfg.window == "head_tail":
        n_head = int(round(L * cfg.head_frac))
        n_tail = L - n_head
        h_full = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)
        # A document that already fits must NOT be spliced, or the head and the
        # tail slice would both cover it and the model would see it twice.
        if len(w) <= budget_words and len(h_full) <= L:
            ids = h_full
        else:
            t = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)
            ids = h_full[:n_head] + t[-n_tail:]
    else:
        raise ValueError(f"unknown window {cfg.window!r}")

    n = max(len(ids), 1)
    if len(ids) < L:
        ids = ids + [pad_id] * (L - len(ids))
    return ids[:L], min(n, L)


def encode_split(tok, rows, pad_id, cfg: LSIConfig, desc=""):
    X = np.full((len(rows), cfg.max_len), pad_id, dtype=np.int32)
    n = np.zeros(len(rows), dtype=np.int32)
    t0 = time.time()
    for i, (_, text, _) in enumerate(rows):
        ids, k = encode_window(tok, text, pad_id, cfg)
        X[i] = ids; n[i] = k
        if desc and i % 2000 == 0 and i:
            print(f"  {desc} {i}/{len(rows)} ({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {len(rows)} docs in {time.time()-t0:.0f}s")
    return X, n


def prepare_data(cfg: LSIConfig, tok, pad_id, splits=("train", "dev"),
                 local_paths: dict | None = None, n_labels=None):
    """Tokenizes once and caches to Drive, keyed by the window settings, so every
    later experiment and every later session reuses it."""
    local_paths = local_paths or {}
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    key = (f"{cfg.window}_{cfg.max_len}_{cfg.head_frac:.2f}"
           f"_ph{int(cfg.collapse_placeholders)}")
    out = {}
    for sp in splits:
        cap = cfg.max_train if sp == "train" else cfg.max_dev
        tag = f"{sp}_{key}" + (f"_n{cap}" if cap else "")
        fx = Path(cfg.cache_dir) / f"X_{tag}.npy"
        fn = Path(cfg.cache_dir) / f"n_{tag}.npy"
        fy = Path(cfg.cache_dir) / f"Y_{tag}.npy"
        fm = Path(cfg.cache_dir) / f"meta_{tag}.json"
        if fx.exists() and fn.exists() and fm.exists():
            meta = json.loads(fm.read_text())
            Y = np.load(fy) if fy.exists() else None
            out[sp] = (np.load(fx), np.load(fn), Y, meta["ids"], meta["labels"])
            print(f"[cache hit ] {sp}: {out[sp][0].shape[0]} docs  ({fx.name})")
            continue
        rows = load_split(cfg, sp, local_paths.get(sp))
        if cap and len(rows) > cap:
            rng = np.random.default_rng(cfg.seed)
            sel = rng.choice(len(rows), cap, replace=False)
            rows = [rows[i] for i in sorted(sel)]
        print(f"[tokenizing] {sp}: {len(rows)} docs, window={cfg.window}, "
              f"max_len={cfg.max_len}")
        X, n = encode_split(tok, rows, pad_id, cfg, desc=sp)
        labs = [r[2] for r in rows]
        Y = None if labs[0] is None else multi_hot(rows, n_labels or cfg.n_labels)
        ids = [r[0] for r in rows]
        np.save(fx, X); np.save(fn, n)
        if Y is not None:
            np.save(fy, Y)
        fm.write_text(json.dumps({"ids": ids, "labels": labs}))
        out[sp] = (X, n, Y, ids, labs)
    return out


class LSIDataset(torch.utils.data.Dataset):
    """Yields (ids, n_real, multi_hot_row, index). The index lets the distillation
    loop look up cached teacher logits without re-running the teacher."""
    def __init__(self, X, n, Y=None):
        self.X, self.n, self.Y = X, n, Y
        self.L = 0 if Y is None else Y.shape[1]
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        y = (torch.from_numpy(self.Y[i]) if self.Y is not None
             else torch.zeros(self.L))
        return torch.from_numpy(self.X[i].astype(np.int64)), int(self.n[i]), y, i


def make_loader(X, n, Y, batch_size, shuffle, seed=42, workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(
        LSIDataset(X, n, Y), batch_size=batch_size, shuffle=shuffle,
        num_workers=workers, pin_memory=(DEVICE.type == "cuda"), generator=g,
        drop_last=False)


# ---- chunking, for the long-document representation study in section 5 ----

def chunk_ids(tok, text, pad_id, L, max_chunks=8, max_words=30000):
    """Evenly spaced L-token chunks spanning the WHOLE document, start to end.
    Short documents give one chunk; long ones give up to max_chunks."""
    w = text.split()[:max_words]
    ids = tok.encode(" ".join(w), add_special_tokens=True)
    T = len(ids)
    if T <= L:
        starts = [0]
    else:
        k = min(math.ceil(T / L), max_chunks)
        starts = [0] if k == 1 else [int(round(i * (T - L) / (k - 1)))
                                     for i in range(k)]
    out = []
    for s in starts:
        c = ids[s:s + L]
        out.append(c + [pad_id] * (L - len(c)))
    return out


def encode_chunks_split(tok, rows, pad_id, L, max_chunks=8, desc=""):
    X, doc = [], []
    t0 = time.time()
    for i, (_, text, _) in enumerate(rows):
        for c in chunk_ids(tok, text, pad_id, L, max_chunks):
            X.append(c); doc.append(i)
        if desc and i and i % 400 == 0:
            print(f"  {desc} {i}/{len(rows)} docs ({time.time()-t0:.0f}s)")
    X = np.asarray(X, dtype=np.int32); doc = np.asarray(doc, dtype=np.int64)
    if desc:
        print(f"  {desc}: {len(rows)} docs -> {len(X)} chunks "
              f"(mean {len(X)/max(len(rows),1):.2f}/doc) in {time.time()-t0:.0f}s")
    return X, doc

### Library 3/8 — the Phase 1 backbones and the multi-label classifier

Copied verbatim from the Phase 1 notebook (and used unchanged in Phases 2–4) so the
Phase 1 `state_dict` loads into exactly the code that produced it. The **only** new
piece is `LSIClassifier`: the same mean-pooled readout as Phase 4, with
`n_labels` independent output units instead of two mutually exclusive ones.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight


def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if padding_mask is not None:
            w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))


class BackboneCfg:
    pass


class LegalEncoder(nn.Module):
    """Phase 1 encoder without the MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)   # only matters for tensors absent from the ckpt

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table ({self.rope_cos.size(0)}). "
                f"Phase 1 precomputed RoPE at max_seq_len only; longer context needs "
                f"the buffers rebuilt and position interpolation.")
        x = self.drop(self.tok_emb(ids))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)

    def forward_embeds(self, emb, padding_mask=None):
        """Same stack, entered at the embedding layer. Prompt tuning needs this:
        the learned vectors are not token ids, so they cannot go through tok_emb."""
        T = emb.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"prompt + text = {T} positions exceeds the RoPE table "
                f"({self.rope_cos.size(0)}) — shorten the text window by the number "
                f"of virtual tokens.")
        x = self.drop(emb)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)


# ───────────────────────────── causal decoder (Mistral-style: GQA + RoPE +
# SwiGLU + RMSNorm), copied from the Phase 1 decoder notebook.

def apply_rope_off(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], -1)
    s_e = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], -1)
    return x * c_e + rot * s_e


class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh=cfg.n_query_heads; self.nkh=cfg.n_kv_heads; self.ng=cfg.n_groups
        self.hd=cfg.head_dim; self.dm=cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope_off(q, cos, sin, offset)
        k = apply_rope_off(k, cos, sin, offset)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, c = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        return x + self.ffn(self.norm2(x)), c


class LegalDecoder(nn.Module):
    """Phase 1 causal decoder, used here as a feature extractor."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.rope_table_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(LegalEncoder._init)

    def _causal_mask(self, T, device):
        cols = torch.arange(T, device=device)[None, :]
        rows = torch.arange(T, device=device)[:, None]
        m = torch.zeros((T, T), device=device).masked_fill(cols > rows, float("-inf"))
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(f"sequence length {T} exceeds the RoPE table "
                             f"({self.rope_cos.size(0)}).")
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device)
        for layer in self.layers:
            x, _ = layer(x, self.rope_cos, self.rope_sin, mask, None, 0)
        return self.norm(x)

    def forward_embeds(self, emb, padding_mask=None):
        T = emb.size(1)
        x = self.drop(emb)
        mask = self._causal_mask(T, emb.device)
        for layer in self.layers:
            x, _ = layer(x, self.rope_cos, self.rope_sin, mask, None, 0)
        return self.norm(x)


def backbone_cfg_from_sd(sd, n_heads=12, max_seq_len=512, dropout=0.1,
                         rope_theta=10000.0):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_heads, c.max_seq_len, c.dropout = n_heads, max_seq_len, dropout
    c.rope_theta = rope_theta
    if c.d_model % c.n_heads:
        raise ValueError(f"d_model {c.d_model} not divisible by n_heads {c.n_heads}")
    c.head_dim = c.d_model // c.n_heads
    return c


def decoder_cfg_from_sd(sd, n_query_heads=12, max_seq_len=512, dropout=0.1,
                        rope_theta=10000.0, rope_table_len=None):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_query_heads = n_query_heads
    c.head_dim = c.d_model // n_query_heads
    c.n_kv_heads = sd["layers.0.attn.k.weight"].shape[0] // c.head_dim
    if c.n_query_heads % c.n_kv_heads:
        raise ValueError(f"n_query_heads {c.n_query_heads} not divisible by "
                         f"inferred n_kv_heads {c.n_kv_heads}")
    c.n_groups = c.n_query_heads // c.n_kv_heads
    c.n_heads = c.n_query_heads
    c.max_seq_len, c.dropout, c.rope_theta = max_seq_len, dropout, rope_theta
    c.rope_table_len = rope_table_len or max(max_seq_len, 2 * max_seq_len)
    return c


class LSIClassifier(nn.Module):
    """Phase 1 backbone + masked mean-pool + one linear layer with `n_labels`
    independent logits (binary relevance). The readout is the SAME in every
    experiment, so a difference in score comes from the adaptation technique and
    not from the head — which the rules say does not count as a technique."""
    def __init__(self, backbone, d_model, n_labels, p_drop=0.1, pool="mean"):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.drop = nn.Dropout(p_drop)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02)
        # A negative bias starts every label at a low probability, which matches a
        # multi-label problem where each document carries a handful of the labels
        # rather than half of them, and stops the first epoch from being spent
        # unlearning an all-positive prediction.
        nn.init.constant_(self.head.bias, -2.0)
        self.is_causal = isinstance(backbone, LegalDecoder)

    def encode(self, ids, pad_mask):
        if self.is_causal:
            return self.backbone(ids)          # causal: no padding mask needed
        return self.backbone(ids, pad_mask)

    def pooled(self, ids, pad_mask):
        h = self.encode(ids, pad_mask)
        if self.is_causal and self.pool == "last":
            lens = (~pad_mask).sum(1).clamp(min=1)
            return h[torch.arange(h.size(0), device=h.device), lens - 1]
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        return (h * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.pooled(ids, pad_mask)))


def _load_ckpt_sd(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = {(k[7:] if k.startswith("module.") else k): v
          for k, v in (ck["model"] if "model" in ck else ck).items()}
    return ck, sd


def load_phase1_model(cfg: LSIConfig, n_layers_keep: int | None = None,
                      verbose=True):
    """Builds the multi-label classifier on whichever Phase 1 model cfg.backbone
    names. n_layers_keep < depth takes an evenly spaced subset of blocks, which is
    how the distillation student is built."""
    is_dec = cfg.backbone == "decoder"
    ck, sd = _load_ckpt_sd(cfg.dec_ckpt if is_dec else cfg.enc_ckpt)

    # ---- is this file really the model cfg.backbone claims? ----
    d = sd["tok_emb.weight"].shape[1]
    kw = sd.get("layers.0.attn.k.weight")
    if is_dec and ("mlm_dense.weight" in sd or "mlm_head.weight" in sd):
        raise ValueError("that file has an MLM head — it is the MHA ENCODER "
                         "checkpoint; set BACKBONE='encoder'")
    if not is_dec:
        if "lm_head.weight" in sd and "mlm_dense.weight" not in sd:
            raise ValueError("that file is a causal DECODER checkpoint; set "
                             "BACKBONE='decoder'")
        if kw is not None and kw.shape[0] != d:
            raise ValueError(f"that file has grouped KV heads (k is {tuple(kw.shape)}, "
                             f"not {d}x{d}) — it is the GQA decoder, not the encoder")

    if is_dec:
        bc = decoder_cfg_from_sd(sd, n_query_heads=cfg.n_query_heads,
                                 max_seq_len=cfg.max_len, dropout=cfg.dropout,
                                 rope_theta=cfg.rope_theta,
                                 rope_table_len=cfg.rope_table_len)
    else:
        bc = backbone_cfg_from_sd(sd, n_heads=cfg.n_heads, max_seq_len=cfg.max_len,
                                  dropout=cfg.dropout, rope_theta=cfg.rope_theta)

    keep_idx = list(range(bc.n_layers))
    if n_layers_keep is not None and n_layers_keep < bc.n_layers:
        step = bc.n_layers / n_layers_keep
        keep_idx = [min(bc.n_layers - 1, int(round(i * step)))
                    for i in range(n_layers_keep)]
        bc.n_layers = n_layers_keep

    bb = LegalDecoder(bc) if is_dec else LegalEncoder(bc)
    want, new = bb.state_dict(), {}
    for k in want:
        if k.startswith("layers."):
            i = int(k.split(".")[1])
            src = k.replace(f"layers.{i}.", f"layers.{keep_idx[i]}.", 1)
        else:
            src = k
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    missing = [k for k in want if k not in new]
    bb.load_state_dict(new, strict=False)

    if verbose:
        kind = ("GQA DECODER (causal, next-token-pretrained)" if is_dec
                else "MHA ENCODER (bidirectional, MLM-pretrained)")
        print(f"{kind}: {bc.d_model}d x {bc.n_layers}L, d_ff={bc.d_ff}, "
              f"vocab={bc.vocab_size:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors (step {ck.get('step','?')}, "
              f"val_loss {ck.get('val_loss', float('nan')):.4f})")
        if keep_idx != list(range(len(keep_idx))):
            print(f"  layer subset: {keep_idx}")
        if missing:
            print(f"  NOT loaded (fresh init): {missing}")
        std = bb.tok_emb.weight.std().item()
        print(f"  tok_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check that the checkpoint really loaded")
        print(f"  head: {cfg.n_labels} independent statute logits")

    model = LSIClassifier(bb, bc.d_model, cfg.n_labels, cfg.dropout,
                          pool=(cfg.dec_pool if is_dec else "mean"))
    # keep only metadata: the Phase 1 checkpoint also carries its AdamW state
    # (~1 GB for 130M params) which would otherwise sit in RAM for the whole run
    ck = {k: v for k, v in ck.items() if k not in ("model", "opt", "scaler")}
    del sd
    return model, bc, ck

### Library 4/8 — PEFT, written from scratch

Ported from **Assignment 4 Q4** (soft prompts 4.1, bottleneck adapters 4.2, LoRA 4.3,
LoRMA 4.4) and **Q5.1** (NF4 weight quantization). The one thing that changes is the
orientation trap: GPT-2's `Conv1D` stores `weight` as `[in, out]` and computes
`x @ W + b`, whereas the Phase 1 backbone uses plain `nn.Linear`, whose weight is
`[out, in]` and which computes `x @ Wᵀ`. Every factor shape below follows the
`nn.Linear` convention, so the A4 code is transposed where it needs to be and **not**
copied blindly.

Nothing here imports `peft`, `bitsandbytes` or any adapter library.

| module | equation | trainable per wrapped `nn.Linear` |
|---|---|---|
| `LoRALinear` | `h = W₀x + (α/r)·B A x` | `r(d_in + d_out)` |
| `LoRMALinear` (LoRMA₊) | `h = ((α/r)·B A + I)·W₀x` | `2r·d_out` |
| `BottleneckAdapter` | `h' = h + W_up GELU(W_down h + b_down) + b_up` | `2·d·b + b + d` |
| `SoftPrompt` | prepend `m` learned vectors to the embeddings | `m·d` |
| `NF4Linear` | frozen `W₀` stored in 4-bit, dequantized in the forward | 0 |

In [ ]:
# ════════════════════════════════════════════ bookkeeping

def param_report(model, label=""):
    tot = sum(p.numel() for p in model.parameters())
    # a 4-bit weight is still a weight: NF4Linear holds it in a uint8 buffer, not a
    # Parameter, so without this the QLoRA row would claim a model 7x smaller than
    # it is and its "% trainable" would be meaningless
    tot += sum(m.n for m in model.modules() if m.__class__.__name__ == "NF4Linear")
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}trainable {tr:,} / {tot:,} = {100*tr/max(tot,1):.3f}%")
    return tr, tot


def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def get_head(model):
    """The classification head, wherever it sits (a soft-prompt model wraps the
    classifier, so the head is one level down)."""
    return model.head if hasattr(model, "head") else model.clf.head


def unfreeze_head(model):
    for p in get_head(model).parameters():
        p.requires_grad = True


def trainable_names(model):
    return [n for n, p in model.named_parameters() if p.requires_grad]


# ════════════════════════════════════════════ 4.3  LoRA

class LoRALinear(nn.Module):
    """A4 Q4.3, in nn.Linear orientation.  h = W₀x + (α/r)·B A x
    W₀ frozen; A [r, d_in] Kaiming-uniform, B [d_out, r] zeros, so ΔW = 0 at step 0
    and training starts at exactly the pretrained function. `swap_ab` is A4's
    Experiment 1 (A = 0, B random): also a zero update at step 0, but the first
    gradient lands on the other factor."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0, swap_ab=False):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.alpha, self.swap_ab = r, alpha, swap_ab
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        if swap_ab:
            nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))   # A stays 0
        else:
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))   # B stays 0
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.T) @ self.lora_B.T * self.scaling

    @torch.no_grad()
    def merged_copy(self):
        """An independent nn.Linear with the scaled update applied exactly once.
        Merge into a COPY, never into the shared pretrained object."""
        new = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None)
        new.weight.copy_(self.base.weight + self.scaling * (self.lora_B @ self.lora_A))
        if self.base.bias is not None:
            new.bias.copy_(self.base.bias)
        return new


# ════════════════════════════════════════════ 4.4  LoRMA (multiplicative)

LORMA_VARIANT = {
    "name": "LoRMA+",
    "equation": "h = ((alpha/r) * B A + I_dout) @ W0 @ x   (additive rank inflation)",
    "paper_version": "Bihany, Patel & Modi, arXiv:2506.07621v1, Eq. 8",
}


class LoRMALinear(nn.Module):
    """A4 Q4.4 — LoRMA₊, multiplicative adaptation with additive rank inflation.

    LoRA ADDS a low-rank matrix to W₀; LoRMA MULTIPLIES W₀ by one. A naive
    (BA)·W₀ has rank ≤ r and so cannot even represent "no change", because I has
    full rank — that is the rank-inhibition problem. LoRMA₊ restores full rank by
    adding the identity:

        W' = ((α/r)·B A + I_{d_out}) · W₀,     B ∈ R^{d_out×r},  A ∈ R^{r×d_out}

    The forward below is the **re-ordered** form: it never materialises the
    d_out × d_out product, it applies the factors to the activations instead —
        h = y + (α/r)·B(A y),   y = W₀x
    which is the same function at a fraction of the cost. `merged_copy` is the
    dense form, kept for the equality check and for merged-latency measurement.

    Initialisation: B = 0 with A random, so W' = W₀ exactly at step 0 (the paper's
    choice). `swap_ab=True` is A4's Experiment 1 — A = 0, B random — which is also
    exactly W₀ at step 0, but puts the first gradient on the other factor."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0, swap_ab=False):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base.out_features
        self.r, self.alpha, self.swap_ab = r, alpha, swap_ab
        self.scaling = alpha / r
        self.lorma_A = nn.Parameter(torch.zeros(r, d_out))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))
        if swap_ab:
            nn.init.kaiming_uniform_(self.lorma_B, a=math.sqrt(5))  # A stays 0
        else:
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))  # B stays 0
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):                      # re-ordered form
        y = self.base(x)
        return y + (self.drop(y) @ self.lorma_A.T) @ self.lorma_B.T * self.scaling

    @torch.no_grad()
    def merged_copy(self):                     # dense form: W' = (sBA + I) W0
        d_out = self.base.out_features
        M = self.scaling * (self.lorma_B @ self.lorma_A) + torch.eye(
            d_out, device=self.lorma_B.device, dtype=self.base.weight.dtype)
        new = nn.Linear(self.base.in_features, d_out,
                        bias=self.base.bias is not None)
        new.weight.copy_(M @ self.base.weight)
        if self.base.bias is not None:
            new.bias.copy_(M @ self.base.bias)
        return new


def lorma_rank_report(module: LoRMALinear, tol=1e-6):
    """Numerical ranks via SVD: the factors, the naive product B A (rank ≤ r), and
    the inflated transformation (α/r)BA + I (full rank). This is the evidence for
    the 4.4(a) argument, and it is worth printing once in the report."""
    with torch.no_grad():
        BA = (module.lorma_B @ module.lorma_A).double()
        M = module.scaling * BA + torch.eye(BA.size(0), dtype=torch.float64)
        W0 = module.base.weight.double()
        dW = (M @ W0) - W0
        rk = lambda T: int((torch.linalg.svdvals(T) > tol).sum())
        return {"r": module.r, "rank_B": rk(module.lorma_B.double()),
                "rank_A": rk(module.lorma_A.double()), "rank_BA": rk(BA),
                "rank_inflated": rk(M), "rank_update_W": rk(dW),
                "d_out": BA.size(0)}


# ════════════════════════════════════════════ 4.2  bottleneck adapters

class BottleneckAdapter(nn.Module):
    """A4 Q4.2.  h' = h + W_up GELU(W_down h + b_down) + b_up

    W_up and b_up are zero-initialised, so the adapted model is EXACTLY the frozen
    model at step 0. `enabled=False` returns the input unchanged, which is how the
    layer-ablation study switches a trained adapter off without retraining."""
    def __init__(self, d_model, bottleneck=32, init_std=None):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        if init_std is None:
            nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
            nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        else:                                   # A4's Gaussian-init variant
            nn.init.normal_(self.down.weight, std=init_std); nn.init.zeros_(self.down.bias)
            nn.init.normal_(self.up.weight, std=init_std); nn.init.zeros_(self.up.bias)
        self.enabled = True

    def forward(self, h):
        if not self.enabled:
            return h
        return h + self.up(F.gelu(self.down(self.norm(h))))


class AdaptedBlock(nn.Module):
    """Frozen encoder block, one adapter on its output (A4's single post-block
    placement; Houlsby et al. use two per block, after each sublayer)."""
    def __init__(self, block: EncoderBlock, d_model, bottleneck=32, init_std=None):
        super().__init__()
        self.block = block
        self.adapter = BottleneckAdapter(d_model, bottleneck, init_std)
    def forward(self, x, cos, sin, padding_mask=None):
        return self.adapter(self.block(x, cos, sin, padding_mask))


class AdaptedDecoderBlock(nn.Module):
    """Same, for the decoder block — whose forward returns (hidden, kv_cache)."""
    def __init__(self, block, d_model, bottleneck=32, init_std=None):
        super().__init__()
        self.block = block
        self.adapter = BottleneckAdapter(d_model, bottleneck, init_std)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, c = self.block(x, cos, sin, mask, kv_cache, offset)
        return self.adapter(h), c


# ════════════════════════════════════════════ 4.1  soft prompts

class SoftPrompt(nn.Module):
    """m trainable continuous vectors, prepended to the input embeddings."""
    def __init__(self, init_embeddings):          # [m, d]
        super().__init__()
        self.prompt = nn.Parameter(init_embeddings.detach().clone().float())
    def forward(self, token_embeddings, pad_mask):
        B = token_embeddings.size(0)
        p = self.prompt.unsqueeze(0).expand(B, -1, -1).to(token_embeddings.dtype)
        m = self.prompt.size(0)
        new_mask = torch.cat(
            [torch.zeros(B, m, dtype=pad_mask.dtype, device=pad_mask.device),
             pad_mask], dim=1)                    # prompt positions are never PAD
        return torch.cat([p, token_embeddings], dim=1), new_mask


class SoftPromptClassifier(nn.Module):
    """Frozen backbone + learned prompt + the same head.

    Gradients must flow THROUGH the frozen backbone to reach the prompt — wrapping
    the backbone in torch.no_grad() would be wrong and would train nothing. Pooling
    deliberately covers the real text tokens only: a prompt position is not
    evidence about the document, and including it would let the model answer from
    the prompt alone."""
    def __init__(self, clf: LSIClassifier, soft_prompt: SoftPrompt):
        super().__init__()
        self.clf, self.soft = clf, soft_prompt
    @property
    def backbone(self):
        return self.clf.backbone
    def forward(self, ids, pad_mask):
        bb = self.clf.backbone
        emb = bb.tok_emb(ids)
        emb, mask = self.soft(emb, pad_mask)
        h = bb.forward_embeds(emb, mask)
        m = self.soft.prompt.size(0)
        keep = (~mask).unsqueeze(-1).to(h.dtype)
        keep[:, :m] = 0.0                          # exclude the prompt positions
        pooled = (h * keep).sum(1) / keep.sum(1).clamp(min=1)
        return self.clf.head(self.clf.drop(pooled))


def build_soft_prompt(model: LSIClassifier, n_virtual=16, init="vocab",
                      init_token_ids=None, seed=42):
    """init: 'vocab'  -> the embeddings of n_virtual real vocabulary tokens
             'random' -> uniform[-0.5, 0.5], as in Lester et al.
       Lester et al. report that vocabulary init beats random at small scale,
       which is the regime this backbone is in, so 'vocab' is the default."""
    emb = model.backbone.tok_emb.weight
    if init == "random":
        g = torch.Generator().manual_seed(seed)
        init_emb = (torch.rand(n_virtual, emb.size(1), generator=g) - 0.5)
    else:
        if init_token_ids is None:
            # mid-frequency ids: not the specials at the bottom of the vocab and
            # not the long tail of near-unseen pieces at the top
            start = min(1000, emb.size(0) - n_virtual - 1)
            init_token_ids = list(range(start, start + n_virtual))
        init_emb = emb[torch.tensor(init_token_ids)]
    return SoftPrompt(init_emb)


# ════════════════════════════════════════════ 5.1  NF4 quantization (for QLoRA)

# The 16 NF4 levels: quantiles of a standard normal, normalised to [-1, 1], with an
# exact zero. A4 Q5.1 derives these; they are reproduced here as a constant.
NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0], dtype=torch.float32)


def nf4_quantize(W: torch.Tensor, block=64):
    """Blockwise absmax NF4. Returns (packed uint8 [n/2], scales fp16 [n_blocks],
    shape). Two 4-bit indices share one byte, so the stored tensor really is 4 bits
    per weight and the memory number in the report is honest."""
    flat = W.detach().float().reshape(-1)
    n = flat.numel()
    pad = (-n) % block
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device)])
    blocks = flat.view(-1, block)
    scales = blocks.abs().amax(dim=1).clamp(min=1e-12)
    norm = blocks / scales[:, None]
    idx = (norm[..., None] - NF4_LEVELS.to(W.device)).abs().argmin(-1).to(torch.uint8)
    idx = idx.reshape(-1)
    packed = (idx[0::2] << 4) | idx[1::2]          # two nibbles per byte
    return packed, scales.half(), tuple(W.shape), n, block


def nf4_dequantize(packed, scales, shape, n, block, device=None, dtype=torch.float32):
    hi = (packed >> 4).to(torch.long)
    lo = (packed & 0x0F).to(torch.long)
    idx = torch.stack([hi, lo], dim=1).reshape(-1)
    vals = NF4_LEVELS.to(packed.device)[idx].view(-1, block)
    out = (vals * scales.float()[:, None]).reshape(-1)[:n]
    return out.view(shape).to(dtype)


class NF4Linear(nn.Module):
    """A frozen nn.Linear whose weight lives in 4 bits and is dequantized inside
    the forward. It holds no trainable parameters, which is the point: QLoRA = this
    plus a LoRA branch in full precision on top."""
    def __init__(self, base: nn.Linear, block=64):
        super().__init__()
        packed, scales, shape, n, blk = nf4_quantize(base.weight, block)
        self.register_buffer("packed", packed)
        self.register_buffer("scales", scales)
        self.shape, self.n, self.block = shape, n, blk
        self.in_features, self.out_features = base.in_features, base.out_features
        if base.bias is not None:
            self.register_buffer("bias", base.bias.detach().clone())
        else:
            self.bias = None
        with torch.no_grad():
            W = nf4_dequantize(packed, scales, shape, n, blk)
            self.quant_err = float((W - base.weight).abs().mean())
            self.rel_err = float((W - base.weight).norm() / base.weight.norm())

    def stored_bytes(self):
        return self.packed.numel() + self.scales.numel() * 2

    def forward(self, x):
        W = nf4_dequantize(self.packed, self.scales, self.shape, self.n,
                           self.block, dtype=x.dtype)
        return F.linear(x, W, self.bias)


# ════════════════════════════════════════════ injection helpers

ATTN_TARGETS = ("q", "k", "v", "o")
FFN_TARGETS = ("gate", "up", "down")


def _iter_blocks(model):
    bb = model.backbone if hasattr(model, "backbone") else model.clf.backbone
    return bb.layers


def inject_lowrank(model, cls, r=8, alpha=16, targets=ATTN_TARGETS,
                   ffn_targets=(), p_drop=0.0, swap_ab=False):
    """Replaces the named nn.Linear modules inside each block with `cls`
    (LoRALinear or LoRMALinear). Returns the list of injected modules."""
    injected = []
    for blk in _iter_blocks(model):
        for name in targets:
            new = cls(getattr(blk.attn, name), r, alpha, p_drop, swap_ab)
            setattr(blk.attn, name, new); injected.append(new)
        for name in ffn_targets:
            new = cls(getattr(blk.ffn, name), r, alpha, p_drop, swap_ab)
            setattr(blk.ffn, name, new); injected.append(new)
    return injected


def inject_adapters(model, bottleneck=32, init_std=None):
    bb = model.backbone if hasattr(model, "backbone") else model.clf.backbone
    d = bb.cfg.d_model
    cls = AdaptedDecoderBlock if isinstance(bb, LegalDecoder) else AdaptedBlock
    bb.layers = nn.ModuleList([cls(b, d, bottleneck, init_std) for b in bb.layers])
    return [b.adapter for b in bb.layers]


def quantize_backbone_nf4(model, targets=ATTN_TARGETS, ffn_targets=FFN_TARGETS,
                          block=64):
    """Replaces the frozen projections with NF4 storage. Call BEFORE inject_lowrank
    so the LoRA branch wraps the quantized layer — that is what makes it QLoRA and
    not 'LoRA with a quantized sibling'."""
    q, fp_bytes, q_bytes, errs = [], 0, 0, []
    for blk in _iter_blocks(model):
        for holder, names in ((blk.attn, targets), (blk.ffn, ffn_targets)):
            for name in names:
                base = getattr(holder, name)
                fp_bytes += base.weight.numel() * 4
                new = NF4Linear(base, block)
                q_bytes += new.stored_bytes()
                errs.append(new.rel_err)
                setattr(holder, name, new); q.append(new)
    print(f"NF4: {len(q)} projections  {fp_bytes/1e6:.1f} MB fp32 -> "
          f"{q_bytes/1e6:.1f} MB packed  ({fp_bytes/max(q_bytes,1):.2f}x smaller), "
          f"mean relative weight error {np.mean(errs):.4f}")
    return q, {"fp32_mb": fp_bytes / 1e6, "nf4_mb": q_bytes / 1e6,
               "ratio": fp_bytes / max(q_bytes, 1), "rel_err": float(np.mean(errs))}


def set_partial_trainable(model, last_k=4, train_norm=True, train_embeddings=False):
    """Layer-wise fine-tuning: unfreeze the top `last_k` blocks (plus the final
    norm and the head), leaving the lower blocks and the embedding table frozen."""
    freeze_all(model)
    unfreeze_head(model)
    bb = model.backbone if hasattr(model, "backbone") else model.clf.backbone
    n = len(bb.layers)
    for i in range(max(0, n - last_k), n):
        for p in bb.layers[i].parameters():
            p.requires_grad = True
    if train_norm:
        for p in bb.norm.parameters():
            p.requires_grad = True
    if train_embeddings:
        bb.tok_emb.weight.requires_grad = True
    return list(range(max(0, n - last_k), n))

### Library 5/8 — multi-label loss, metrics and threshold selection

Three pieces that single-label Phase 4 did not need:

* **loss** — binary cross-entropy over `n_labels` independent sigmoids, optionally
  with a per-label `pos_weight`, focal down-weighting, or the asymmetric loss. The
  positive/negative imbalance here is severe (a document carries a handful of the
  statutes and is negative for all the rest), so the loss is a real design choice,
  not a default.
* **metrics** — macro-F1 over *every* label column, micro-F1, subset accuracy. The
  macro average is computed over the full label dimension, including labels that
  never occur in dev, because that is how the evaluator computes it.
* **thresholds** — a sigmoid gives a score, not a decision. A single global cut at
  0.5 is almost never the macro-F1 optimum when labels differ in frequency by two
  orders of magnitude, so this section tunes a global threshold and then per-label
  thresholds on dev, and never on test.

In [ ]:
# ════════════════════════════════════════════ losses

def make_pos_weight(Y, cap=20.0):
    """N_neg / N_pos per label, capped. Without the cap a statute with three
    positives in the whole training set gets a weight in the thousands and drags
    every gradient with it."""
    pos = Y.sum(0)
    neg = len(Y) - pos
    w = np.where(pos > 0, neg / np.maximum(pos, 1.0), 1.0)
    return torch.tensor(np.minimum(w, cap), dtype=torch.float32)


def multilabel_loss(logits, target, cfg: LSIConfig, pos_weight=None):
    if cfg.loss == "bce":
        return F.binary_cross_entropy_with_logits(logits, target)
    if cfg.loss == "bce_posw":
        return F.binary_cross_entropy_with_logits(logits, target,
                                                  pos_weight=pos_weight)
    p = torch.sigmoid(logits)
    if cfg.loss == "focal":
        pt = p * target + (1 - p) * (1 - target)
        ce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
        return ((1 - pt).pow(cfg.focal_gamma) * ce).mean()
    if cfg.loss == "asl":
        # asymmetric loss: harder down-weighting of easy negatives than of
        # positives, plus a probability margin that ignores very confident negatives
        pm = (p - cfg.asl_clip).clamp(min=0)
        l_pos = target * torch.log(p.clamp(min=1e-8)) * (1 - p).pow(cfg.asl_gamma_pos)
        l_neg = (1 - target) * torch.log((1 - pm).clamp(min=1e-8)) * pm.pow(cfg.asl_gamma_neg)
        return -(l_pos + l_neg).mean()
    raise ValueError(f"unknown loss {cfg.loss!r}")


# ════════════════════════════════════════════ metrics (vectorised, no sklearn in
# the inner loop — the threshold search calls these thousands of times)

def per_label_f1(Y, P):
    """Y, P: [N, L] 0/1 arrays. Returns f1 per label (zero_division=0)."""
    Y = Y.astype(np.float64); P = P.astype(np.float64)
    tp = (Y * P).sum(0); fp = ((1 - Y) * P).sum(0); fn = (Y * (1 - P)).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)


def multilabel_metrics(Y, P):
    f1 = per_label_f1(Y, P)
    tp = (Y * P).sum(); fp = ((1 - Y) * P).sum(); fn = (Y * (1 - P)).sum()
    micro = 2 * tp / max(2 * tp + fp + fn, 1e-12)
    prec = tp / max(tp + fp, 1e-12)
    rec = tp / max(tp + fn, 1e-12)
    subset = float((P.astype(np.int8) == Y.astype(np.int8)).all(1).mean())
    return {"macro_f1": float(f1.mean()), "micro_f1": float(micro),
            "precision": float(prec), "recall": float(rec),
            "subset_accuracy": subset,
            "labels_per_doc": float(P.sum(1).mean()),
            "f1_per_label": f1}


# ════════════════════════════════════════════ decision rule

def apply_thresholds(prob, thr, min_labels=1):
    """thr: scalar or [L]. Guarantees at least `min_labels` statutes per document
    by falling back to the top-scoring label(s) — the submission format expects a
    set of statute ids, and an empty set can only lose recall."""
    P = (prob >= np.asarray(thr)).astype(np.int8)
    if min_labels:
        short = P.sum(1) < min_labels
        if short.any():
            rows = np.nonzero(short)[0]
            top = np.argsort(-prob[rows], axis=1)[:, :min_labels]
            for j in range(top.shape[1]):
                P[rows, top[:, j]] = 1
    return P


def tune_global_threshold(prob, Y, cfg: LSIConfig):
    grid = np.linspace(cfg.thr_grid_lo, cfg.thr_grid_hi, cfg.thr_grid_n)
    best, best_t = -1.0, 0.5
    for t in grid:
        f = per_label_f1(Y, apply_thresholds(prob, t, cfg.min_labels)).mean()
        if f > best:
            best, best_t = f, float(t)
    return best_t, float(best)


def tune_per_label_thresholds(prob, Y, cfg: LSIConfig, base_t=None):
    """One threshold per label, each chosen to maximise that label's own F1.

    A label with very few dev positives keeps the global threshold: fitting a cut
    to three examples is fitting noise, and because macro-F1 weights every label
    equally that noise would be worth as much as a well-estimated label. The
    `per_label_min_pos` guard is what keeps this a calibration step rather than a
    way of memorising the dev set."""
    if base_t is None:
        base_t, _ = tune_global_threshold(prob, Y, cfg)
    L = Y.shape[1]
    thr = np.full(L, base_t, dtype=np.float64)
    grid = np.linspace(cfg.thr_grid_lo, cfg.thr_grid_hi, cfg.thr_grid_n)
    pos = Y.sum(0)
    tuned = 0
    for j in range(L):
        if pos[j] < cfg.per_label_min_pos:
            continue
        y = Y[:, j]; p = prob[:, j]
        f_base = per_label_f1(y[:, None], (p >= base_t).astype(np.int8)[:, None])[0]
        best, bt = f_base, base_t
        for t in grid:
            pred = (p >= t).astype(np.int8)
            tp = (y * pred).sum(); fp = ((1 - y) * pred).sum(); fn = (y * (1 - pred)).sum()
            den = 2 * tp + fp + fn
            f = 0.0 if den == 0 else 2 * tp / den
            if f > best:
                best, bt = f, float(t)
        if bt != base_t:
            thr[j] = bt; tuned += 1
    return thr, tuned


def score_with_strategies(prob, Y, cfg: LSIConfig):
    """macro-F1 under the three decision rules, on the same probabilities. This is
    the table that answers 'how do threshold-selection strategies affect
    precision, recall and Macro-F1'."""
    out = {}
    P = apply_thresholds(prob, 0.5, cfg.min_labels)
    out["fixed_0.5"] = {**multilabel_metrics(Y, P), "thr": 0.5}
    gt, _ = tune_global_threshold(prob, Y, cfg)
    P = apply_thresholds(prob, gt, cfg.min_labels)
    out["global_tuned"] = {**multilabel_metrics(Y, P), "thr": gt}
    pt, n_tuned = tune_per_label_thresholds(prob, Y, cfg, gt)
    P = apply_thresholds(prob, pt, cfg.min_labels)
    out["per_label"] = {**multilabel_metrics(Y, P), "thr": pt, "n_tuned": n_tuned}
    for v in out.values():
        v.pop("f1_per_label", None)
    return out, gt, pt

### Library 6/8 — one training loop, used by every experiment

Every experiment calls the same `train_model`. What changes between them is only
which parameters carry `requires_grad`, and (for distillation) the loss. Keeping the
loop fixed is what makes the comparison in section 6 a comparison of *techniques*.

In [ ]:
@torch.no_grad()
def predict_probs(model, loader, pad_id, return_logits=False):
    model.eval()
    out = []
    for ids, n, y, i in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        pm = ids == pad_id
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            logits = model(ids, pm)
        out.append(logits.float().cpu())
    logits = torch.cat(out).numpy()
    return logits if return_logits else 1.0 / (1.0 + np.exp(-logits))


def evaluate(model, loader, Y, pad_id, cfg: LSIConfig, tune=True):
    prob = predict_probs(model, loader, pad_id)
    if tune:
        t, f = tune_global_threshold(prob, Y, cfg)
    else:
        t = 0.5
        f = per_label_f1(Y, apply_thresholds(prob, t, cfg.min_labels)).mean()
    m = multilabel_metrics(Y, apply_thresholds(prob, t, cfg.min_labels))
    m["thr"] = t
    m["_prob"] = prob
    return m


def _param_groups(model, cfg: LSIConfig):
    decay, no_decay, head = [], [], []
    for n_, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n_.startswith("head.") or ".head." in n_:
            head.append(p)
        elif p.ndim < 2 or "norm" in n_.lower() or n_.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay, "weight_decay": cfg.weight_decay, "lr": cfg.lr},
            {"params": no_decay, "weight_decay": 0.0, "lr": cfg.lr},
            {"params": head, "weight_decay": cfg.weight_decay, "lr": cfg.head_lr}]


def train_model(model, train_loader, dev_loader, Ydev, pad_id, cfg: LSIConfig,
                pos_weight=None, loss_fn=None, extra_loss=None, label="run"):
    """loss_fn(logits, target, idx) overrides the task loss (distillation uses it).
    Returns the best-dev state loaded back into `model`."""
    model.to(DEVICE)
    groups = [g for g in _param_groups(model, cfg) if len(g["params"])]
    if not groups:
        raise RuntimeError("nothing is trainable — check the freeze/unfreeze calls")
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    steps_per_epoch = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    total = steps_per_epoch * cfg.epochs
    warm = max(1, int(cfg.warmup_frac * total))
    base_lrs = [g["lr"] for g in opt.param_groups]

    def lr_at(s):
        if s < warm:
            return (s + 1) / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    pw = None if pos_weight is None else pos_weight.to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    hist, best, best_state, no_imp, step, micro = [], -1.0, None, 0, 0, 0
    t0 = time.time()
    model.train()
    stop = False
    for ep in range(cfg.epochs):
        if stop:
            break
        for ids, n, y, idx in train_loader:
            ids = ids.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pm = ids == pad_id
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                logits = model(ids, pm)
                if loss_fn is not None:
                    loss = loss_fn(logits.float(), y, idx)
                else:
                    loss = multilabel_loss(logits.float(), y, cfg, pw)
                if extra_loss is not None:
                    loss = loss + extra_loss()
                loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()
            micro += 1
            if micro % cfg.grad_accum:
                continue

            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for g in opt.param_groups for p in g["params"]], cfg.max_grad_norm)
            f = lr_at(step)
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * f
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1

            if step % cfg.eval_every == 0 or step == total:
                m = evaluate(model, dev_loader, Ydev, pad_id, cfg)
                hist.append({"step": step, "macro_f1": m["macro_f1"],
                             "micro_f1": m["micro_f1"], "thr": m["thr"]})
                flag = ""
                if m["macro_f1"] > best:
                    best = m["macro_f1"]; no_imp = 0
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in model.state_dict().items()}
                    flag = "  *best*"
                else:
                    no_imp += 1
                print(f"  [{label}] step {step}/{total} ep{ep+1} "
                      f"dev macro-F1 {m['macro_f1']:.4f} micro {m['micro_f1']:.4f} "
                      f"t={m['thr']:.2f}{flag}  ({time.time()-t0:.0f}s)")
                model.train()
                if no_imp >= cfg.patience:
                    print(f"  [{label}] early stop: {cfg.patience} evals without "
                          f"improvement")
                    stop = True
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    peak = (torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0)
    return {"best_dev_macro_f1": best, "history": hist,
            "train_seconds": time.time() - t0, "steps": step,
            "peak_train_mem_mb": peak}


# ════════════════════════════════════════════ frozen-feature shortcut
#
# When the backbone is frozen, its output for a given document NEVER CHANGES.
# Running it once per epoch recomputes identical numbers: for the linear probe
# that is the entire cost of the experiment. These two helpers run the backbone
# once, cache the pooled vectors to Drive, and train the head on them — minutes
# instead of an hour, and the head gets 40 epochs instead of 2, which makes the
# floor a fair floor rather than an undertrained one.
#
# This works ONLY for the probe. Every other experiment changes the backbone's
# function, so its features have to be recomputed and the full loop is required.

@torch.no_grad()
def cache_features(model, loader, pad_id, cfg: LSIConfig, tag, desc=""):
    """Pooled [N, d_model] representations, computed once and stored on Drive."""
    p = Path(cfg.cache_dir) / f"feat_{cfg.backbone}_{tag}.npy"
    if p.exists():
        F = np.load(p)
        print(f"[cache hit ] features {F.shape}  ({p.name})")
        return F
    model.to(DEVICE).eval()          # eval(): dropout off, so the cache is
    out, t0 = [], time.time()        # deterministic and reusable
    for ids, n, y, i in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            h = model.pooled(ids, ids == pad_id)
        out.append(h.float().cpu())
    F = torch.cat(out).numpy()
    np.save(p, F)
    print(f"  cached {desc or tag} features {F.shape} in {time.time()-t0:.0f}s "
          f"-> {p.name}  ({F.nbytes/1e6:.0f} MB)")
    return F


def train_head_on_features(model, Ftr, Ytr, Fdv, Ydv, cfg: LSIConfig,
                           pos_weight=None, epochs=40, lr=3e-3, bs=256,
                           label="probe"):
    """Trains model.head on cached features, then leaves it in place so the model
    is an ordinary LSIClassifier that saves, reloads and submits like any other."""
    head = model.head.to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr,
                            weight_decay=cfg.weight_decay)
    Xt, Yt = torch.from_numpy(Ftr), torch.from_numpy(Ytr)
    Xd = torch.from_numpy(Fdv).to(DEVICE)
    pw = None if pos_weight is None else pos_weight.to(DEVICE)
    N, spe = len(Xt), math.ceil(len(Xt) / bs)
    hist, best, best_sd, no_imp = [], -1.0, None, 0
    t0 = time.time()
    g = torch.Generator().manual_seed(cfg.seed)
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(N, generator=g)
        for i in range(0, N, bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), Yt[idx].to(DEVICE)
            loss = multilabel_loss(head(xb), yb, cfg, pw)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            prob = torch.sigmoid(head(Xd)).float().cpu().numpy()
        t, _ = tune_global_threshold(prob, Ydv, cfg)
        m = multilabel_metrics(Ydv, apply_thresholds(prob, t, cfg.min_labels))
        hist.append({"step": (ep + 1) * spe, "macro_f1": m["macro_f1"],
                     "micro_f1": m["micro_f1"], "thr": t})
        if m["macro_f1"] > best:
            best, no_imp = m["macro_f1"], 0
            best_sd = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        else:
            no_imp += 1
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"  [{label}] epoch {ep+1}/{epochs} dev macro-F1 "
                  f"{m['macro_f1']:.4f} micro {m['micro_f1']:.4f} t={t:.2f}"
                  f"{'  *best*' if no_imp == 0 else ''}  ({time.time()-t0:.0f}s)")
        if no_imp >= 8:
            print(f"  [{label}] early stop at epoch {ep+1}")
            break
    if best_sd is not None:
        head.load_state_dict(best_sd)
    return {"best_dev_macro_f1": best, "history": hist,
            "train_seconds": time.time() - t0, "steps": len(hist) * spe,
            "peak_train_mem_mb": (torch.cuda.max_memory_allocated() / 1e6
                                  if DEVICE.type == "cuda" else 0.0)}


# ════════════════════════════════════════════ cost measurement

@torch.no_grad()
def measure_latency(model, pad_id, max_len, batch=16, reps=12, warmup=3):
    model.to(DEVICE).eval()
    ids = torch.full((batch, max_len), 5, dtype=torch.long, device=DEVICE)
    pm = ids == pad_id
    ts = []
    for i in range(reps + warmup):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            model(ids, pm)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        if i >= warmup:
            ts.append(time.time() - t0)
    return float(np.median(ts))


def state_bytes(sd):
    return int(sum(v.numel() * v.element_size() for v in sd.values()
                   if torch.is_tensor(v)))


def trainable_state(model):
    keep = set(trainable_names(model))
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in keep}


# ════════════════════════════════════════════ saving

def save_model(cfg: LSIConfig, name, model, bc, technique, peft_cfg, dev_metrics,
               thresholds, train_info, trainable, total, extra=None):
    """One .pt per experiment, holding everything `rebuild_from_checkpoint` needs
    to reconstruct the exact function — including the PEFT configuration, so the
    rebuild does not have to guess the structure from tensor names."""
    path = Path(cfg.out_dir) / "models" / f"{name}.pt"
    path.parent.mkdir(parents=True, exist_ok=True)
    gt, pt = thresholds
    ck = {"model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
          "backbone": {k: v for k, v in vars(bc).items()
                       if isinstance(v, (int, float, str, bool, type(None)))},
          "technique": technique, "peft": peft_cfg, "name": name,
          "n_labels": cfg.n_labels, "window": cfg.window, "max_len": cfg.max_len,
          "head_frac": cfg.head_frac, "backbone_kind": cfg.backbone,
          "collapse_placeholders": cfg.collapse_placeholders,
          "dec_pool": cfg.dec_pool, "loss": cfg.loss,
          "dev_macro_f1": dev_metrics["macro_f1"],
          "global_thr": float(gt), "per_label_thr": np.asarray(pt).tolist(),
          "history": train_info.get("history", []),
          "trainable": int(trainable), "total": int(total)}
    if extra:
        ck.update(extra)
    torch.save(ck, path)
    print(f"saved {path}  ({path.stat().st_size/1e6:.1f} MB)")
    return str(path)


def save_dev_analysis(cfg: LSIConfig, name, ids, prob, Y, meta):
    """The .npz/.json pair the analysis section reads, so section 6 never has to
    re-run a model."""
    d = Path(cfg.out_dir) / "analysis"
    d.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(d / f"{name}.npz", ids=np.array(ids), prob=prob, Y=Y)
    (d / f"{name}.json").write_text(json.dumps(meta, indent=1, default=float))


def record_result(cfg: LSIConfig, name, technique, strategies, trainable, total,
                  train_info, costs, notes="", extra=None):
    path = Path(cfg.results_json)
    allr = json.loads(path.read_text()) if path.exists() else []
    allr = [r for r in allr if r.get("name") != name]
    best = strategies["per_label"]
    row = {"name": name, "technique": technique,
           "macro_f1": best["macro_f1"], "micro_f1": best["micro_f1"],
           "precision": best["precision"], "recall": best["recall"],
           "subset_accuracy": best["subset_accuracy"],
           "labels_per_doc": best["labels_per_doc"],
           "macro_f1_at_half": strategies["fixed_0.5"]["macro_f1"],
           "macro_f1_global_thr": strategies["global_tuned"]["macro_f1"],
           "global_thr": strategies["global_tuned"]["thr"],
           "n_labels_tuned": best.get("n_tuned", 0),
           "trainable_params": int(trainable), "total_params": int(total),
           "trainable_pct": round(100 * trainable / max(total, 1), 4),
           "train_seconds": round(train_info.get("train_seconds", 0), 1),
           "peak_train_mem_mb": round(train_info.get("peak_train_mem_mb", 0), 1),
           "steps": train_info.get("steps", 0),
           "history": train_info.get("history", []),
           "window": cfg.window, "max_len": cfg.max_len, "epochs": cfg.epochs,
           "lr": cfg.lr, "loss": cfg.loss,
           "batch": cfg.batch_size * cfg.grad_accum, "quick": cfg.quick,
           "notes": notes, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    row.update(costs or {})
    if extra:
        row.update(extra)
    allr.append(row)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(allr, indent=1, default=float))
    print(f"recorded '{name}' -> {path}")
    return row


def load_results(cfg: LSIConfig):
    path = Path(cfg.results_json)
    return json.loads(path.read_text()) if path.exists() else []


def already_done(cfg: LSIConfig, name):
    """Idempotence: an experiment that is already recorded AND has its .pt on Drive
    is skipped, so re-running the notebook after a disconnect costs nothing."""
    has_row = any(r.get("name") == name for r in load_results(cfg))
    has_ckpt = (Path(cfg.out_dir) / "models" / f"{name}.pt").exists()
    if has_row and has_ckpt:
        row = next(r for r in load_results(cfg) if r["name"] == name)
        print(f"SKIP {name} — already on Drive "
              f"(dev macro-F1 {row['macro_f1']:.4f}). Delete its row in "
              f"results.json to re-run.")
        return True
    return False


# ════════════════════════════════════════════ rebuilding and submitting

def rebuild_from_checkpoint(cfg: LSIConfig, path, dropout=0.0, verbose=True):
    """Rebuilds any model this notebook saved, from its .pt alone: the stored
    `peft` dict says what to re-inject, then the load is strict. The caller
    re-checks dev macro-F1 against the value saved at training time, so a wrong
    reconstruction cannot pass silently."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck["model"]
    bc = BackboneCfg(); bc.__dict__.update(ck["backbone"]); bc.dropout = dropout
    is_dec = "n_kv_heads" in ck["backbone"]
    if is_dec and getattr(bc, "rope_table_len", None) is None:
        bc.rope_table_len = 2 * bc.max_seq_len
    bb = LegalDecoder(bc) if is_dec else LegalEncoder(bc)
    model = LSIClassifier(bb, bc.d_model, ck["n_labels"], dropout,
                          pool=(ck.get("dec_pool", "last") if is_dec else "mean"))

    p = ck.get("peft") or {}
    kind = p.get("kind", "plain")
    if p.get("nf4"):
        # the quantization scope is its own thing: QLoRA quantizes the whole
        # backbone but puts the low-rank branch on the attention projections only,
        # so reusing the LoRA targets here would leave the FFN unquantized and the
        # state_dict would not fit
        quantize_backbone_nf4(model, tuple(p.get("nf4_targets", ATTN_TARGETS)),
                              tuple(p.get("nf4_ffn_targets", FFN_TARGETS)))
    if kind == "lora":
        inject_lowrank(model, LoRALinear, p["r"], p["alpha"],
                       tuple(p.get("targets", ATTN_TARGETS)),
                       tuple(p.get("ffn_targets", ())), swap_ab=p.get("swap_ab", False))
    elif kind == "lorma":
        inject_lowrank(model, LoRMALinear, p["r"], p["alpha"],
                       tuple(p.get("targets", ATTN_TARGETS)),
                       tuple(p.get("ffn_targets", ())), swap_ab=p.get("swap_ab", False))
    elif kind == "adapter":
        inject_adapters(model, p["bottleneck"], p.get("init_std"))
    elif kind == "soft_prompt":
        model = SoftPromptClassifier(
            model, build_soft_prompt(model, p["n_virtual"], p.get("init", "vocab")))

    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [m for m in missing if "rope_" not in m]
    if missing or unexpected:
        raise RuntimeError(f"{Path(path).name}: structure mismatch\n"
                           f"  missing {missing[:6]}\n  unexpected {unexpected[:6]}")
    model.eval()
    meta = {"name": ck.get("name", Path(path).stem),
            "technique": ck.get("technique", "?"),
            "backbone": "decoder" if is_dec else "encoder",
            "n_layers": bc.n_layers, "window": ck.get("window", "head_tail"),
            "max_len": ck.get("max_len", bc.max_seq_len),
            "head_frac": ck.get("head_frac", 0.5),
            "collapse_placeholders": ck.get("collapse_placeholders", True),
            "saved_dev_macro_f1": ck.get("dev_macro_f1"),
            "global_thr": ck.get("global_thr", 0.5),
            "per_label_thr": np.asarray(ck.get("per_label_thr", [])),
            "kind": kind, "trainable": ck.get("trainable"),
            "total": ck.get("total"), "history": ck.get("history")}
    if verbose:
        print(f"{meta['name']}: {meta['backbone']} {bc.n_layers}L, {kind}, "
              f"window={meta['window']} max_len={meta['max_len']}, "
              f"saved dev macro-F1 {meta['saved_dev_macro_f1']}")
    return model, meta


def prepare_test(wcfg: LSIConfig, tok, pad_id, local_test=None):
    """Tokenized test set, cached on Drive per window setting. Several runs
    usually share a window, and without this each one would re-tokenize the whole
    test set from scratch."""
    tag = (f"test_{wcfg.window}_{wcfg.max_len}_{wcfg.head_frac:.2f}"
           f"_ph{int(wcfg.collapse_placeholders)}")
    fx = Path(wcfg.cache_dir) / f"X_{tag}.npy"
    fn = Path(wcfg.cache_dir) / f"n_{tag}.npy"
    fi = Path(wcfg.cache_dir) / f"ids_{tag}.json"
    if fx.exists() and fn.exists() and fi.exists():
        ids = json.loads(fi.read_text())
        print(f"[cache hit ] test: {len(ids)} docs  ({fx.name})")
        return np.load(fx), np.load(fn), ids
    rows = load_split(wcfg, "test", local_test)
    print(f"[tokenizing] test: {len(rows)} docs, window={wcfg.window}, "
          f"max_len={wcfg.max_len}")
    X, n = encode_split(tok, rows, pad_id, wcfg, desc="test")
    ids = [r[0] for r in rows]
    Path(wcfg.cache_dir).mkdir(parents=True, exist_ok=True)
    np.save(fx, X); np.save(fn, n); fi.write_text(json.dumps(ids))
    return X, n, ids


def write_submission(cfg: LSIConfig, model, meta, tok, pad_id, name,
                     thr=None, local_test=None, batch_size=16, valid_ids=None):
    """Writes submissions/{name}/prediction.json  =  [{"id": ..., "pred": [ints]}]
    and zips it. Every guarantee the task statement asks for is asserted here:
    integers only, valid statute ids only, unique within an example, one object per
    test document."""
    wcfg = LSIConfig(**{**asdict(cfg), "window": meta["window"],
                        "max_len": meta["max_len"], "head_frac": meta["head_frac"],
                        "collapse_placeholders": meta["collapse_placeholders"]})
    X, n, test_ids = prepare_test(wcfg, tok, pad_id, local_test)
    loader = make_loader(X, n, None, batch_size, shuffle=False, workers=0)
    prob = predict_probs(model, loader, pad_id)

    if thr is None:
        thr = meta["per_label_thr"] if len(np.asarray(meta["per_label_thr"])) \
              else meta["global_thr"]
    if valid_ids is not None:
        # a column that never carried a positive in train or dev is not a statute
        # the evaluator can credit, so it is zeroed BEFORE thresholding — that way
        # the "at least one label" fallback also picks from the valid ids only
        mask = np.zeros(prob.shape[1], dtype=np.float32)
        mask[list(valid_ids)] = 1.0
        prob = prob * mask[None, :]
    P = apply_thresholds(prob, thr, cfg.min_labels)

    sub = [{"id": test_ids[i], "pred": sorted(int(j) for j in np.nonzero(P[i])[0])}
           for i in range(len(test_ids))]
    assert len(sub) == len(test_ids), "one prediction object per test document"
    for s in sub:
        assert all(isinstance(v, int) for v in s["pred"]), "integers only"
        assert len(set(s["pred"])) == len(s["pred"]), "unique ids within an example"
        assert len(s["pred"]) >= 1, "at least one statute id"
        if valid_ids is not None:
            assert set(s["pred"]) <= set(valid_ids), "valid statute ids only"

    d = Path(cfg.out_dir) / "submissions" / name
    d.mkdir(parents=True, exist_ok=True)
    jp = d / "prediction.json"
    jp.write_text(json.dumps(sub, indent=1))
    zp = d / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, "prediction.json")
    cnt = Counter(len(s["pred"]) for s in sub)
    print(f"wrote {jp}\n      {zp}")
    print(f"  {len(sub)} rows | mean {np.mean([len(s['pred']) for s in sub]):.2f} "
          f"statutes/doc | size histogram {dict(sorted(cnt.items())[:8])}")
    return sub

def write_all_submissions(cfg: LSIConfig, tok, pad_id, names=None,
                          valid_ids=None, skip_existing=True, local_test=None):
    """One submission zip per recorded run, so every experiment can go on the
    leaderboard instead of only the best one.

    Each run is rebuilt from its own checkpoint, re-checked against the dev score
    it was saved with, and predicted with its OWN window and its OWN dev-tuned
    thresholds — a run trained on a shorter window is tokenized accordingly rather
    than being forced through the best run's settings. Returns a table sorted by
    dev macro-F1, which is the order to upload in."""
    rows = load_results(cfg)
    if names is None:
        names = [r["name"] for r in rows if not r["name"].startswith("S")]
    rows = {r["name"]: r for r in rows}
    out = []
    for nm in names:
        d = Path(cfg.out_dir) / "submissions" / nm
        if skip_existing and (d / "submission.zip").exists():
            print(f"SKIP {nm} — submissions/{nm}/submission.zip already written")
            out.append({"name": nm, "dev_macro_f1": rows.get(nm, {}).get("macro_f1"),
                        "zip": str(d / "submission.zip"), "status": "cached"})
            continue
        ck = Path(cfg.out_dir) / "models" / f"{nm}.pt"
        if not ck.exists():
            print(f"SKIP {nm} — no checkpoint at {ck}")
            continue
        print(f"\n─── {nm} " + "─" * (56 - len(nm)))
        try:
            model, meta = rebuild_from_checkpoint(cfg, ck)
            model.to(DEVICE).eval()
            sub = write_submission(cfg, model, meta, tok, pad_id, nm,
                                   valid_ids=valid_ids, local_test=local_test)
            out.append({"name": nm,
                        "technique": rows.get(nm, {}).get("technique", "?"),
                        "dev_macro_f1": rows.get(nm, {}).get("macro_f1"),
                        "mean_labels": float(np.mean([len(s["pred"]) for s in sub])),
                        "zip": str(d / "submission.zip"), "status": "written"})
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            out.append({"name": nm, "status": f"failed: {type(e).__name__}"})
    out.sort(key=lambda r: -(r.get("dev_macro_f1") or -1))
    return out

### Library 7/8 — plots

One palette, fixed order, used everywhere, so a technique keeps its colour from
figure to figure. The palette is colour-blind-safe at adjacent pairs; every bar is
also directly labelled with its value, so no reading depends on colour alone.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID = "#1a1a19", "#5c5b55", "#dcdbd4"

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "font.size": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
})

COLOR_OF = {}            # technique -> colour, assigned once, never cycled


def color_for(name):
    if name not in COLOR_OF:
        COLOR_OF[name] = PALETTE[len(COLOR_OF) % len(PALETTE)]
    return COLOR_OF[name]


def savefig(fig, cfg: LSIConfig, name, show=True):
    p = Path(cfg.out_dir) / "plots" / f"{name}.png"
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    print(f"plot -> {p}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return str(p)

## 3 — Data, label space, and the experiment runner

In [ ]:
cfg = LSIConfig(
    drive_dir=DRIVE, enc_ckpt=ENC_PATH, dec_ckpt=DEC_PATH,
    tokenizer_json=TOK_JSON_PATH, out_dir=OUT_DIR,
    cache_dir=f"{OUT_DIR}/cache", results_json=f"{OUT_DIR}/results.json",
    backbone=BACKBONE, max_len=MAX_LEN, rope_table_len=ROPE_TABLE_LEN,
    hf_token=HF_TOKEN, quick=QUICK,
)
set_seed(cfg.seed)
tok, pad_id, base_vocab = load_tokenizer(cfg)
print(f"tokenizer: vocab {base_vocab:,}, pad id {pad_id}")

# ---- label space, from train + dev ----
train_rows = load_split(cfg, "train")
dev_rows   = load_split(cfg, "dev")
if cfg.max_train and len(train_rows) > cfg.max_train:
    rng = np.random.default_rng(cfg.seed)
    train_rows = [train_rows[i] for i in
                  sorted(rng.choice(len(train_rows), cfg.max_train, replace=False))]
if cfg.max_dev and len(dev_rows) > cfg.max_dev:
    rng = np.random.default_rng(cfg.seed)
    dev_rows = [dev_rows[i] for i in
                sorted(rng.choice(len(dev_rows), cfg.max_dev, replace=False))]

VALID_IDS, N_LABELS = label_space(train_rows, dev_rows)
cfg.n_labels = N_LABELS

# ════════════ tokenizer reality check ════════════
# Everything here is measured with YOUR a1_tokenizer + bpe_40k.json, so the numbers
# that go in the report are this tokenizer's, not an estimate from the vocabulary
# file. Costs a few seconds on a 300-document sample.
print("\n--- tokenizer reality check (your a1_tokenizer, 300-doc sample) ---")
_sample = train_rows[:300]
_words = sum(len(r[1].split()) for r in _sample)
_toks = sum(len(tok.encode(r[1], add_special_tokens=False)) for r in _sample)
TOK_PER_WORD = _toks / max(_words, 1)
print(f"tokens per word: {TOK_PER_WORD:.3f}")

for orig, sym in PLACEHOLDER_MAP.items():
    a = len(tok.encode(" " + orig, add_special_tokens=False))
    b = len(tok.encode(" " + sym, add_special_tokens=False))
    print(f"  {orig:<11} {a} token(s)   ->  {sym!r} {b} token(s)")

if cfg.collapse_placeholders:
    # exact: count the symbols already substituted and price each at the
    # difference between the original string's cost and the symbol's
    _saved = 0
    for orig, sym in PLACEHOLDER_MAP.items():
        _c = sum(r[1].count(sym) for r in _sample)
        _cost = (len(tok.encode(" " + orig, add_special_tokens=False))
                 - len(tok.encode(" " + sym, add_special_tokens=False)))
        _saved += _c * _cost
    if _saved:
        print(f"placeholder collapsing frees "
              f"{100*_saved/max(_toks+_saved,1):.1f}% of the token budget — the "
              f"same {cfg.max_len}-token window covers "
              f"{(_toks+_saved)/max(_toks,1):.2f}x more text. Section 5.2b trains "
              f"the uncollapsed side so this is a measurement, not a guess.")
else:
    print("placeholder collapsing is OFF (cfg.collapse_placeholders=False)")

# how much of a document a single window can actually see
_wl = np.array([len(r[1].split()) for r in train_rows])
_tl = _wl * TOK_PER_WORD
print(f"\ndocument length: median {np.median(_tl):.0f} tokens, mean {_tl.mean():.0f}, "
      f"p90 {np.percentile(_tl,90):.0f}, max {_tl.max():.0f}")
for _n in (cfg.max_len, 2*cfg.max_len, 4*cfg.max_len):
    print(f"  a {_n}-token window holds the WHOLE document for "
          f"{( _tl <= _n).mean():.1%} of the training set")
if (_tl <= cfg.max_len).mean() < 0.7:
    print(f"So for {(_tl > cfg.max_len).mean():.0%} of documents the window is a "
          f"sample, not the document. The eight experiments deliberately share one "
          f"window so the comparison isolates the adaptation technique — but the "
          f"window may well be what caps the score, which is what section 5.2 "
          f"and 5.3 are for. Run them early, not last.")

# ---- tokenize + cache (this is the slow part, once per window setting) ----
data = prepare_data(cfg, tok, pad_id, ("train", "dev"), n_labels=N_LABELS)
Xtr, ntr, Ytr, ids_tr, labs_tr = data["train"]
Xdv, ndv, Ydv, ids_dv, labs_dv = data["dev"]

assert ids_dv == [r[0] for r in dev_rows], \
    "the cached dev order and the in-memory dev rows disagree — clear cache/"
assert ids_tr == [r[0] for r in train_rows], \
    "the cached train order and the in-memory train rows disagree — clear cache/"

TRAIN_LOADER = make_loader(Xtr, ntr, Ytr, cfg.batch_size, True, cfg.seed)
DEV_LOADER   = make_loader(Xdv, ndv, Ydv, 16, False, cfg.seed)
POS_WEIGHT   = make_pos_weight(Ytr, cfg.pos_weight_cap)

# ---- what the label distribution actually looks like ----
freq = Ytr.sum(0)
per_doc = Ytr.sum(1)
print(f"\ntrain {len(Xtr):,} docs | dev {len(Xdv):,} docs | {N_LABELS} statute slots")
print(f"statutes per document: mean {per_doc.mean():.2f}, median "
      f"{np.median(per_doc):.0f}, max {per_doc.max():.0f}, "
      f"{(per_doc > 1).mean():.1%} of documents carry more than one")
print(f"label frequency in train: min {freq.min():.0f}, median "
      f"{np.median(freq):.0f}, max {freq.max():.0f} "
      f"(imbalance ratio {freq.max()/max(freq.min(),1):.0f}x)")
q33, q66 = np.quantile(freq[freq > 0], [0.33, 0.66])
RARE   = np.where((freq > 0) & (freq <= q33))[0]
MEDIUM = np.where((freq > q33) & (freq <= q66))[0]
COMMON = np.where(freq > q66)[0]
NEVER  = np.where(freq == 0)[0]
print(f"frequency terciles: least-frequent {len(RARE)} labels "
      f"(<= {freq[RARE].max() if len(RARE) else 0:.0f} train docs) | "
      f"middle {len(MEDIUM)} | most-frequent {len(COMMON)} "
      f"(>= {freq[COMMON].min() if len(COMMON) else 0:.0f}) | "
      f"unseen in train {len(NEVER)}")
print(f"positive rate overall: {Ytr.mean():.4%}  "
      f"-> pos_weight (capped at {cfg.pos_weight_cap:g}) median "
      f"{np.median(POS_WEIGHT.numpy()):.1f}")

# ---- what the label distribution means for this phase ----
print()
if freq.min() >= 100:
    print(f"NOTE: the rarest statute still has {freq.min():.0f} training documents, so "
          f"this is a MODERATELY imbalanced problem, not a long tail. Do not write "
          f"'rare statutes' in the report as though some were few-shot — the honest "
          f"claim is a {freq.max()/max(freq.min(),1):.0f}x frequency ratio across "
          f"terciles. It also means per-label thresholds are well estimated: every "
          f"label has plenty of dev positives.")
else:
    print(f"The rarest statute has only {freq.min():.0f} training documents. Macro-F1 "
          f"weights it the same as the most frequent, so the least-frequent tercile "
          f"decides the score.")
if len(NEVER):
    print(f"{len(NEVER)} label column(s) never occur in train+dev; they contribute 0 "
          f"to macro-F1 no matter what the model does.")

# ---- is the dense label matrix a problem? (the sparse-storage question) ----
dense_mb = Ytr.nbytes / 1e6
nnz = int(Ytr.sum())
print(f"\nmulti-hot label matrix: {Ytr.shape[0]}x{Ytr.shape[1]} float32 = "
      f"{dense_mb:.0f} MB dense, {100*nnz/Ytr.size:.2f}% non-zero "
      f"(a CSR/sparse version would be ~{(nnz*8 + len(Ytr)*4)/1e6:.1f} MB).")
print(f"Dense is kept: at {N_LABELS} labels the saving is a few MB and sparse rows "
      f"would complicate the DataLoader, the loss and every metric. Sparse storage "
      f"starts to matter in the thousands of labels, not here.")

# ---- how long will this take? ----
steps_ep = math.ceil(len(Xtr) / cfg.batch_size / cfg.grad_accum)
print(f"\nbudget: {steps_ep:,} optimizer steps/epoch x {cfg.epochs} epochs = "
      f"{steps_ep*cfg.epochs:,} steps, {len(Xtr)//cfg.batch_size:,} "
      f"forward/backward passes per epoch at {cfg.max_len} tokens.")
print(f"On a T4 that is roughly 20-30 min per epoch for a full fine-tune, so about "
      f"an hour per experiment and 10-15 hours for the whole phase. If you are short "
      f"of time, set cfg.max_train (the rarest label keeps "
      f"{freq.min():.0f} x fraction documents) or drop to epochs=2 — and say in the "
      f"report that the budget was fixed across techniques, which is what makes the "
      f"comparison fair.")

tokenizer: vocab 40,000, pad id 0


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

statute ids: 100 distinct, min 0, max 99 -> label dimension 100

--- tokenizer reality check (your a1_tokenizer, 300-doc sample) ---
tokens per word: 1.130
  <SECTION>   4 token(s)   ->  '§' 1 token(s)
  <ACT>       4 token(s)   ->  '¶' 1 token(s)
  <ENTITY>    5 token(s)   ->  '@' 1 token(s)
placeholder collapsing frees 18.2% of the token budget — the same 512-token window covers 1.22x more text. Section 5.2b trains the uncollapsed side so this is a measurement, not a guess.

document length: median 486 tokens, mean 1271, p90 3126, max 98162
  a 512-token window holds the WHOLE document for 51.3% of the training set
  a 1024-token window holds the WHOLE document for 66.4% of the training set
  a 2048-token window holds the WHOLE document for 81.1% of the training set
So for 49% of documents the window is a sample, not the document. The eight experiments deliberately share one window so the comparison isolates the adaptation technique — but the window may well be what caps the score, w

In [ ]:
from dataclasses import replace

def finish_experiment(name, technique, model, bc, peft_cfg, ecfg, info, tr, tot,
                      notes="", extra=None, dev_loader=None, Yd=None, ids_d=None):
    """Score on dev under all three decision rules, measure the costs, and write
    the model, the dev predictions and the results row to Drive."""
    dl = dev_loader or DEV_LOADER
    Yd = Ydv if Yd is None else Yd
    ids_d = ids_dv if ids_d is None else ids_d
    prob = predict_probs(model, dl, pad_id)
    strat, gt, pt = score_with_strategies(prob, Yd, ecfg)
    best = strat["per_label"]
    print(f"\n[{name}] dev   macro-F1  t=0.5 {strat['fixed_0.5']['macro_f1']:.4f}"
          f"  |  global t={gt:.2f} {strat['global_tuned']['macro_f1']:.4f}"
          f"  |  per-label {best['macro_f1']:.4f}"
          f"   (micro {best['micro_f1']:.4f}, subset acc {best['subset_accuracy']:.4f})")

    lat = measure_latency(model, pad_id, ecfg.max_len)
    costs = {"latency_s_batch16": round(lat, 4),
             "trainable_state_mb": round(state_bytes(trainable_state(model)) / 1e6, 3),
             "model_mb": round(state_bytes(model.state_dict()) / 1e6, 1)}
    save_model(ecfg, name, model, bc, technique, peft_cfg, best, (gt, pt),
               info, tr, tot, extra=extra)
    save_dev_analysis(ecfg, name, ids_d, prob, Yd,
                      {"name": name, "technique": technique, "peft": peft_cfg,
                       "window": ecfg.window, "max_len": ecfg.max_len,
                       "head_frac": ecfg.head_frac, "loss": ecfg.loss,
                       "trainable": int(tr), "total": int(tot),
                       "global_thr": float(gt),
                       "per_label_thr": np.asarray(pt).tolist(),
                       "strategies": {k: {kk: vv for kk, vv in v.items()
                                          if kk != "thr"}
                                      for k, v in strat.items()},
                       **costs})
    return record_result(ecfg, name, technique, strat, tr, tot, info, costs,
                         notes=notes, extra=extra)


def freeze_and_enable(model, modules, attrs):
    """Freeze everything, then switch on only the PEFT tensors and the head."""
    freeze_all(model)
    unfreeze_head(model)
    for m in modules:
        for a in attrs:
            getattr(m, a).requires_grad_(True)


def best_run(cfg, exclude=("00_head_only", "08b_student_nokd")):
    """The best of the real experiments, read off Drive. The linear probe is a
    floor and the no-KD student is a control, so neither is eligible, and the
    reduced-budget `S*` study runs are not comparable to the full-budget ones."""
    rows = [r for r in load_results(cfg)
            if not r["name"].startswith("S") and r["name"] not in exclude]
    if not rows:
        raise RuntimeError("no experiments recorded yet — run notebooks 01-08")
    return max(rows, key=lambda r: r["macro_f1"])


print("runner ready. Every experiment below is idempotent — re-running the notebook "
      "skips what is already on Drive.")

runner ready. Every experiment below is idempotent — re-running the notebook skips what is already on Drive.


### 4.5 — `05_adapters`: residual bottleneck adapters, from scratch

A4 Q4.2, single post-block placement, bottleneck 32, up-projection zero-initialised
so the adapted model **is** the frozen model at step 0.

In [ ]:
NAME = "05_adapters"
if not already_done(cfg, NAME):
    set_seed(cfg.seed)
    ecfg = replace(cfg, lr=3e-4, head_lr=1e-3)
    peft = {"kind": "adapter", "bottleneck": 32, "init_std": None}
    model, bc, _ = load_phase1_model(ecfg)

    # functional-equality check: the adapter must not change the model at step 0
    model.eval()
    xb, nb, yb, ib = next(iter(DEV_LOADER))
    with torch.no_grad():
        before = model(xb[:4], xb[:4] == pad_id).clone()
    ads = inject_adapters(model, peft["bottleneck"], peft["init_std"])
    model.eval()                       # the freshly built adapters default to train
    with torch.no_grad():
        after = model(xb[:4], xb[:4] == pad_id)
    print(f"  identity at init: max|logit change| = "
          f"{(after - before).abs().max():.2e}  (must be 0)")

    freeze_all(model); unfreeze_head(model)
    for a in ads:
        for p in a.parameters():
            p.requires_grad = True
    tr, tot = param_report(model, f"[{NAME}] ")
    d, b = bc.d_model, peft["bottleneck"]
    hand = len(ads) * (d * b + b + b * d + d + d)   # down, b_down, up, b_up, RMSNorm
    print(f"  hand-derived adapter parameters: {len(ads)} x ({d}*{b} + {b} + "
          f"{b}*{d} + {d} + {d}) = {hand:,}; counted (incl. head) {tr:,}")

    info = train_model(model, TRAIN_LOADER, DEV_LOADER, Ydv, pad_id, ecfg,
                       pos_weight=POS_WEIGHT, label=NAME)
    finish_experiment(NAME, f"bottleneck adapters (b={b}, post-block)", model, bc,
                      peft, ecfg, info, tr, tot,
                      notes="non-mergeable: adds depth at inference")
    del model; torch.cuda.empty_cache()

MHA ENCODER (bidirectional, MLM-pretrained): 768d x 14L, d_ff=2048, vocab=40,001
  loaded 128/128 tensors (step 29000, val_loss 1.6970)
  tok_emb std = 0.0267 (fresh init ~0.0200)
  head: 100 independent statute logits
  identity at init: max|logit change| = 0.00e+00  (must be 0)
[05_adapters] trainable 786,980 / 130,620,452 = 0.602%
  hand-derived adapter parameters: 14 x (768*32 + 32 + 32*768 + 768 + 768) = 710,080; counted (incl. head) 786,980
  [05_adapters] step 300/8016 ep1 dev macro-F1 0.0942 micro 0.1902 t=0.04  *best*  (266s)
  [05_adapters] step 600/8016 ep1 dev macro-F1 0.1709 micro 0.2491 t=0.04  *best*  (531s)
  [05_adapters] step 900/8016 ep1 dev macro-F1 0.1972 micro 0.3399 t=0.08  *best*  (796s)
  [05_adapters] step 1200/8016 ep1 dev macro-F1 0.2225 micro 0.3610 t=0.08  *best*  (1061s)
  [05_adapters] step 1500/8016 ep1 dev macro-F1 0.2441 micro 0.3530 t=0.08  *best*  (1326s)
  [05_adapters] step 1800/8016 ep1 dev macro-F1 0.2390 micro 0.3611 t=0.08  (1591s)
  [05_adapt

---
Next: `lsi_06_soft_prompt`.